# Module 3: LangSmith — Prompt Engineering, Observability & Evaluations

> Part of the **Modular Workshops** series. Standalone, ~30 min.

We evaluate and observe the **internal HR assistant** from Module 1 (all HR data is synthetic — see `utils/hr_seed.py`). We cover four parts plus a closing loop:

1. **Prompt engineering** — author, test, and version prompts in the Playground and Prompt Hub, then pull them into code with the SDK.
2. **Tracing** — generate traces with the HR agent, then query them with `list_runs` + filters.
3. **Offline evaluations** — build a dataset of representative questions + edge cases, then score the agent with four evaluators (figure correctness, tool selection, case-opened-iff-should, and an LLM-judge on helpfulness). Discover evaluators **by tag** and wire a **CI promotion gate**.
4. **Online evaluations** — score every new trace as it lands, plus the minimal path for an app to attach 👍/👎 + a comment.

Then **annotation queues** close the loop: route runs flagged by eval scores to a human for review.

<img src="../images/evals-conceptual.png" style="width: auto; max-height: 400px; border-radius: 8px;">

## Setup


In [1]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.models import model
from utils.langsmith_rules import (
    get_or_create_annotation_queue,
    create_run_rule,
    delete_run_rule,
)

import os, time
from datetime import datetime, timedelta, timezone
from typing_extensions import TypedDict
from langchain_core.messages import SystemMessage, HumanMessage
from langsmith import Client, uuid7

client = Client()
print("LANGSMITH_TRACING:", os.environ.get("LANGSMITH_TRACING", "not set"))
print("Project:", os.environ.get("LANGSMITH_PROJECT", "default"))


LANGSMITH_TRACING: true
Project: hr-agent-workshop-william-reinhard


## Part 1. Prompt Engineering — Playground & Prompt Hub

Before you can observe or evaluate an agent, you need a prompt worth shipping. LangSmith treats prompts as **versioned artifacts** — author and test them in the **Playground**, version and share them in the **Prompt Hub**, then pull them into code with the SDK. Three surfaces, one source of truth.

- **Playground** (UI) — an interactive editor: compose messages, wire up input variables, pick a model, and run.
- **Prompt Hub** (UI) — every saved prompt with full commit history, tags, and a public hub of community prompts to fork.
- **SDK** — `push_prompt` / `pull_prompt` to move prompts between code and the hub.

### 1.1 The Prompt Playground

Open **Prompts** in the LangSmith sidebar and click **+ Prompt** to land in the Playground. The left panel is your prompt — an ordered list of messages, each with a role:

- **System** — the instruction manual: persona and ground rules.
- **Human** — the user's turn.
- **AI** — a model turn, handy for few-shot examples.
- **Tool** — tool output, for testing how the model reacts to it.

Add an input variable by typing `{variable_name}` into any message (or highlight text and click **Convert to variable**). Fill in sample values in the right panel's **Inputs** box, then click **Start** to run and see the response.

**Template format.** Variables default to Python **f-string** syntax (`{topic}`). Switch to **mustache** (`{{topic}}`) from the format dropdown when you need loops, conditionals, or nested data (`{{user.name}}`) — f-strings only do flat substitution.

**Model configuration.** Click the **gear icon** next to the model name to set provider, model, temperature, and max tokens. Hit **Save As** to name a configuration — it's shared across your workspace and reusable in other LangSmith features.

**Tools.** Click **+ Tool** to attach tools: built-in ones (web search, code interpreter) or custom tools you define with a name, description, and argument schema. When the model calls a tool, the Playground shows the tool name and arguments so you can verify the call.

🔗 **Try it:** [Open Prompts in LangSmith →](https://smith.langchain.com/prompts) — then click **+ Prompt** (top right) to open the Playground.

### 1.2 Prompt Hub — save, version, share

Click **Save** in the Playground and your prompt lands in the **Prompts** table. Each prompt gets its own detail page with a two-pane layout: commit history and environments on the left, the selected commit on the right.

- **Commits** — every save is a new commit, and the full history is preserved. Toggle **Diff** (top-right) to compare a commit with its predecessor.
- **Tags** — mark a commit with a stable name (e.g. `prod`) so code can reference it without pinning a hash. Move or delete tags as the prompt evolves.
- **Environments** — reserved **Staging** and **Production** environments track which commit is live; **Promote** a commit to move it forward, or roll back from history.
- **Public hub** — search community prompts by name, use case, or model, and **fork** any of them into your workspace.

🔗 **Open in LangSmith:** [Your prompts →](https://smith.langchain.com/prompts) · [Public LangChain Hub →](https://smith.langchain.com/hub)

### 1.3 Manage prompts programmatically

Anything you do in the UI you can do from the SDK: `push_prompt` sends a prompt to the hub, and `pull_prompt` fetches it back. We'll do it in three quick steps — **push** an HR case-summary prompt, **pull it and run it as an agent** (with `create_agent`, not a raw chain), then **version** it.

In [17]:
from langchain_core.prompts import ChatPromptTemplate

# Step 1 — author the HR assistant's system prompt and push it to the hub.
prompt_name = "hr-case-summary"
prompt = ChatPromptTemplate([
    ("system",
     "You are an internal HR assistant. Look up the employee, then produce a concise "
     "case summary: (1) a one-line status, (2) whether their salary is inside band "
     "(compare to band min/mid/max for their level), (3) a recommended next step, and "
     "(4) whether an HR case should be opened. Treat compensation data as confidential; "
     "never disclose another employee's pay. "),
])

url = client.push_prompt(prompt_name, object=prompt)
print("Prompt page (click to open):", url)

Prompt page (click to open): https://smith.langchain.com/prompts/hr-case-summary/5e9c7ee3?organizationId=e3272f09-bf53-4cd5-a37c-93bd272d3078


**Pull it back and run it — as an agent, not a chain.** `pull_prompt` returns the prompt we just pushed; we use it as the system prompt for `create_agent`, running on the workshop's shared `model`. A small mock `lookup_employee` tool lets the agent fetch the profile, so the full agent loop (model → tool → model) shows up in the trace.

In [18]:
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def lookup_employee(employee_id: int) -> str:
    """Look up an employee's profile and salary band by ID (synthetic data)."""
    # Mock HR lookup — swap for the real DB-backed tool (agents/hr_agent.py) in production.
    directory = {
        1001: (
            "id 1001, Emerson Kim, Associate IT Specialist (IT, L1). Salary $57,000. "
            "Band min/mid/max: 69,000 / 81,000 / 93,000. Manager id 1168. Tenure 5.6y."
        ),
    }
    return directory.get(employee_id, f"No profile found for id {employee_id!r}.")


# Step 2 — pull the prompt back and run it with create_agent (uses the imported `model`).
pulled = client.pull_prompt(prompt_name)
system_prompt = pulled.format_messages()[0].content

agent = create_agent(model=model, tools=[lookup_employee], system_prompt=system_prompt)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Summarize the comp situation for employee 1001."}]}
)
print(result["messages"][-1].text)

**Status:** Employee 1001 is materially below the L1 salary range despite 5.6 years of tenure.

- **Band alignment:** Salary is **$57,000** versus L1 band **$69,000 / $81,000 / $93,000** (min/mid/max), placing them **$12,000 below the band minimum**.
- **Recommended next step:** Review for an off-cycle market/equity adjustment, confirm job level and any applicable pay-policy exceptions, and develop a plan to bring pay to at least the band minimum.
- **Open an HR case:** **Yes** — this is an out-of-band compensation exception requiring review and documentation.


**Version it.** Re-push under the same name and LangSmith records a new commit — the earlier version stays in the history.

In [ ]:
# Step 3 — re-push a tweaked version. Same name -> a new commit (full history preserved).
prompt_v2 = ChatPromptTemplate([
    ("system",
     "You are an internal HR assistant. Look up the employee, then produce a concise "
     "case summary: (1) a one-line status, (2) whether their salary is inside band "
     "(compare to band min/mid/max for their level), (3) a recommended next step, "
     "(4) whether an HR case should be opened, and (5) one data-quality flag if any "
     "figure is missing (e.g. a NULL salary). Treat compensation data as confidential."),
])
url_v2 = client.push_prompt(prompt_name, object=prompt_v2)
print("New commit (click to open):", url_v2)

# Pull a specific commit with client.pull_prompt("hr-case-summary:<commit-hash>"),
# and tear down with client.delete_prompt("hr-case-summary") when you're done.

## Warm-up: Generate a few traces

Before we look at tracing and querying, let's actually produce some traces. 
We invoke the HR assistant from Module 1 (`agents/hr_agent.py`) three times with **deliberately light** prompts — 
each one is a single lookup so the runs finish in a few seconds. (All HR data is synthetic — see `utils/hr_seed.py`.)

On the trial run while building this module the warm-up took **~9 seconds total (3.1s avg per call)**. Expect similar.


In [ ]:
from agents.hr_agent import build_hr_agent

agent = build_hr_agent()

warmup_prompts = [
    "Is employee 1001 paid inside band for their level?",
    "Who reports to manager 1062?",
    "What is the comp band for Software Engineering L3?",
]

total = 0.0
for q in warmup_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].text[:120]}")

print(f"\nTotal: {total:.1f}s ({total/len(warmup_prompts):.1f}s avg)")


## Part 2. Tracing + Querying Traces

Set `LANGSMITH_TRACING=true` and every LLM call, tool call, and state transition lands in your tracing project — no code changes required. 
(The warm-up above already produced traces; this section pulls them back out.)

We use `client.list_runs(...)` to query them.

In [4]:
project_name = os.environ.get("LANGSMITH_PROJECT", "modular-workshops")
try:
    project = client.read_project(project_name=project_name)
    print(f"Project: {project.name}")
    print(f"View traces: {project.url}")
except Exception as e:
    print(f"Could not read project (this is fine if first run): {e}")


Project: hr-agent-workshop-william-reinhard
View traces: https://smith.langchain.com/o/e3272f09-bf53-4cd5-a37c-93bd272d3078/projects/p/2c1c82b4-9604-4900-abbd-ccb46f5354b9


### 2.1 Pull recent traces

Useful filters on `client.list_runs(...)`:

- `project_name=` — scope to one project
- `start_time=` / `end_time=` — time window
- `run_type=` — `"llm"`, `"tool"`, `"chain"`, `"retriever"`
- `error=True` — only failed runs
- `is_root=True` — only top-level traces (not their children)
- `filter=` — LangSmith filter DSL (latency, feedback, attributes...)

In [ ]:
from datetime import datetime, timedelta, timezone

# Pull the last hour of root traces from this workshop's project
since = datetime.now(timezone.utc) - timedelta(hours=1)

recent_runs = list(client.list_runs(
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    start_time=since,
    is_root=True,
    limit=20,
))

print(f"Found {len(recent_runs)} root run(s) in the last hour\n")
for r in recent_runs[:5]:
    latency = (r.end_time - r.start_time).total_seconds() if r.end_time else None
    print(f"- {r.id}  {r.name:25s}  latency={latency}s  error={r.error is not None}")


### 2.2 Filter DSL — find slow or errored runs

The `filter` argument is a small expression language. Common patterns:

- `gt(latency, 5)` — slower than 5 seconds
- `eq(status, "error")` — failed runs
- `and(eq(feedback_key, "correctness"), lt(feedback_score, 0.5))` — low-scored runs on a feedback key
- Combine with `and(...)` / `or(...)`

In [ ]:
# Find slow root runs in the last hour (>5s latency)
slow_runs = list(client.list_runs(
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    start_time=since,
    is_root=True,
    filter='gt(latency, 5)',
    limit=20,
))

print(f"{len(slow_runs)} slow root run(s) (>5s) in the last hour")
for r in slow_runs[:5]:
    latency = (r.end_time - r.start_time).total_seconds()
    print(f"  {r.name:25s}  {latency:.1f}s  {r.id}")


## Part 3. Offline Evaluations

**Offline evals** are the experiments you run on demand against a fixed dataset. Build a dataset once, then score the HR agent against it whenever you change a prompt, a model, or a tool — a clean before/after comparison, and (below) a **CI promotion gate**.

Four pieces:
1. **Dataset** — representative questions + edge cases, with reference answers
2. **Target function** — runs the agent once per example, returning its response *and* its tool trajectory
3. **Evaluators** — four of them, each scoring a different property
4. **A tag registry** — so a pipeline discovers evaluators **by tag** instead of a hardcoded list

### 3.1 Dataset — representative questions + edge cases

The dataset lives in `evals/hr_dataset.py` (15 examples) so the notebook and CI share one source of truth. Each example's reference carries four kinds of ground truth — one per evaluator:

- `reference_answer` — success rubric + key figures (for the LLM judge)
- `expected_figures` — exact figures that must appear in the answer (figure correctness)
- `expected_tools` — the tools the agent should call (tool selection)
- `should_open_case` — whether `open_hr_case` must fire (retrieve-then-**act** guardrail)

The edge cases are all in here: below-band-min (1001), null salary (1004), duplicate name (Lucia Brown → 1002/1003), and a manager with no reports (1201). It also includes *negative* action cases — read-only questions where opening a case would be **over-acting**.

In [ ]:
from evals.hr_dataset import build_dataset, DATASET_NAME, EXAMPLES

dataset = build_dataset(client)   # idempotent: replaces the dataset to match the file
print(f"Created dataset '{dataset.name}' with {len(EXAMPLES)} examples")
print(f"View: {dataset.url}")

# Peek at one edge-case example
print("\nExample (below-band-min):")
print(EXAMPLES[0])

### 3.2 Target function

One agent run per example, returning everything the evaluators need: the final `response` and the ordered `trajectory` of tool calls. It's packaged in `evals/target.py` so CI reuses the exact same target.

In [ ]:
from evals.target import run_hr_agent

# Smoke-test the target on one example before running the full experiment.
demo = run_hr_agent({"query": "Is employee 1001 paid inside band for their level?"})
print("response:", demo["response"][:200])
print("trajectory:", demo["trajectory"])

### 3.3 Four evaluators — one per property

`evals/hr_evaluators.py` defines four evaluators. Three are deterministic/code-based; one is an LLM-as-judge:

| Evaluator | What it checks | Kind |
|---|---|---|
| `figure_correctness` | every expected figure appears in the answer | code |
| `tool_selection` | agent called the expected tools (multiset), reports missing/extra | code |
| `case_opened_correctly` | `open_hr_case` fired **iff** it should have (no over/under-acting) | code |
| `answer_helpfulness` | helpful, factual, appropriately cautious on edge cases | LLM-judge |

Each returns `{"key", "score", "comment"}` — the shape `client.evaluate` expects.

In [ ]:
from evals.hr_evaluators import (
    figure_correctness,
    tool_selection,
    case_opened_correctly,
    answer_helpfulness,
)

# Sanity-check the three code-based evaluators on a fabricated (output, reference) pair.
ref = EXAMPLES[10]["outputs"]   # the "open a comp review case for 1001" example
fake_output = {
    "response": "Employee 1001 earns $57,000, below the band min of $69,000. Opened case #1.",
    "trajectory": ["lookup_employee", "open_hr_case"],
}
print(figure_correctness(EXAMPLES[10]["inputs"], fake_output, ref))
print(tool_selection(EXAMPLES[10]["inputs"], fake_output, ref))
print(case_opened_correctly(EXAMPLES[10]["inputs"], fake_output, ref))

# The guardrail catches over-acting too: a read-only question that opened a case.
ro = EXAMPLES[13]
print(case_opened_correctly(
    ro["inputs"],
    {"response": "1005 is inside band.", "trajectory": ["lookup_employee", "open_hr_case"]},
    ro["outputs"],
))

### 3.4 Discover evaluators **by tag** (customer requirement)

> **Customer requirement:** a pipeline must *discover* which evaluators to run by **tag**, not by a hardcoded list in the workflow script.

Evaluators are registered with tags in `evals/hr_evaluators.py` via a small registry. A pipeline asks the registry for a tag and gets back the callables to run. Below we show the tag lookup **explicitly** — this is exactly what the CI runner does.

In [ ]:
from evals.hr_evaluators import list_tags, get_evaluators_by_tag, get_registered_by_tag

# All tags -> which evaluators carry them.
print("Tag map:")
for tag, keys in list_tags().items():
    print(f"  {tag:12s} -> {keys}")

# EXPLICIT tag lookup: the pipeline selects evaluators by tag, not by name.
CI_TAG = "ci"
ci_registered = get_registered_by_tag(CI_TAG)
ci_evaluators = get_evaluators_by_tag(CI_TAG)   # the callables to pass to client.evaluate

print(f"\nEvaluators discovered for tag '{CI_TAG}': {[r.key for r in ci_registered]}")
print(f"(The LLM-judge 'answer_helpfulness' is tagged 'llm-judge'/'offline', not 'ci'.)")

### 3.5 Run the experiment

We pass the tag-discovered evaluators to `client.evaluate` — plus the LLM judge so the notebook run scores all four. (CI gates only on the deterministic `ci` ones; see 3.6.)

In [ ]:
results = client.evaluate(
    run_hr_agent,
    data=DATASET_NAME,
    # Tag-discovered evaluators + the LLM judge (for the interactive run).
    evaluators=[*ci_evaluators, answer_helpfulness],
    experiment_prefix="hr-offline",
    max_concurrency=4,
)
print(f"View at: {results.experiment_name}")

### 3.6 CI promotion gate

`evals/run_evals.py` is the CI entrypoint. It (1) rebuilds the dataset, (2) **discovers evaluators by tag** (`--tag ci`), (3) runs the experiment, and (4) **exits non-zero if any gated score drops below a configurable threshold** — so a failing job blocks the merge.

```bash
# Run locally exactly as CI does:
python -m evals.run_evals --tag ci --threshold 0.8 \
    --gate-keys figure_correctness,tool_selection,case_opened_correctly
```

Everything is configurable via flags or env vars (`EVAL_TAG`, `EVAL_THRESHOLD`, `EVAL_GATE_KEYS`, ...).

**The workflow** — `.github/workflows/evals.yml` runs this on every pull request that touches `agents/`, `evals/`, or `utils/`, and on manual dispatch with a custom threshold:

```yaml
env:
  EVAL_TAG: ci
  EVAL_THRESHOLD: ${{ github.event.inputs.threshold || '0.8' }}
  EVAL_GATE_KEYS: "figure_correctness,tool_selection,case_opened_correctly"

steps:
  - run: |
      uv run python -m evals.run_evals \
        --tag "$EVAL_TAG" --threshold "$EVAL_THRESHOLD" --gate-keys "$EVAL_GATE_KEYS"
```

The job needs `OPENAI_API_KEY` and `LANGSMITH_API_KEY` as repo secrets. Because the runner reads the threshold and gate keys from env, tightening the gate is a one-line config change — no script edits.

## Part 4. Online Evaluations

**Online evals** run automatically against every new trace as it lands in your tracing project — triggered on incoming runs instead of a dataset.

LangSmith calls these **run rules**. The Python SDK doesn't expose them directly, so we wrap the REST endpoint with a helper at `utils/langsmith_rules.py`.

Online evaluators come in two flavors:

- **LLM-as-judge** — runs a model *on LangSmith's infra*, so it needs a model-provider secret (`OPENAI_API_KEY` / gateway key) stored in **LangSmith → Workspace → Secrets**.
- **Code evaluator** — runs inline Python on LangSmith's infra with **no model and no network**, so it needs **no secret**. Perfect when your workspace has no provider key.

Since this workspace has no model secret, we use a **code evaluator**. It takes one `run` argument and returns a feedback dict (`{key: score}`). Here it does two deterministic checks on every root trace: the assistant returned a substantive answer, and whether the "act" step (`open_hr_case`) fired — so you can spot acting runs in production at a glance.

In [24]:
def _tenant_id(client) -> str:
    return str(client.read_project(project_name=os.environ.get("LANGSMITH_PROJECT", "hr-agent-workshop-william-reinhard")).tenant_id)

def create_run_rule(client, *, project_name, display_name, filter, sampling_rate=1.0,
                    code_evaluator=None, add_to_annotation_queue_id=None,
                    add_to_dataset_id=None):
    project_id = str(client.read_project(project_name=project_name).id)
    rule = {
        "display_name": display_name,
        "sampling_rate": sampling_rate,
        "session_id": project_id,
        "filter": filter,
        "is_enabled": True,
    }
    if code_evaluator is not None:
        rule["code_evaluator"] = code_evaluator
    if add_to_annotation_queue_id is not None:
        rule["add_to_annotation_queue_id"] = str(add_to_annotation_queue_id)
    if add_to_dataset_id is not None:
        rule["add_to_dataset_id"] = str(add_to_dataset_id)
    resp = client.request_with_retries("POST", "/runs/rules", request_kwargs={"json": rule})
    resp.raise_for_status()
    payload = resp.json()
    tenant = payload.get("tenant_id") or _tenant_id(client)
    print(tenant)
    return {
        "id": payload["id"],
        "payload": payload,
        "url": f"https://smith.langchain.com/o/{tenant}/projects/p/{project_id}/rules",
    }

def delete_run_rule(client, rule_id):
    resp = client.request_with_retries("DELETE", f"/runs/rules/{rule_id}")
    resp.raise_for_status()
    return True

In [25]:
# A CODE-based online evaluator: runs on LangSmith's infra, no model/secret needed.
# It receives one `run` dict and returns a feedback dict {key: score}.
# NOTE: this source is executed remotely by LangSmith — keep it self-contained
# (standard library only; no imports from this repo).
code_evaluator_source = """
def perform_eval(run):
    outputs = run.get("outputs") or {}

    # The agent's final text can surface a few ways depending on serialization.
    text = ""
    if isinstance(outputs, dict):
        if isinstance(outputs.get("output"), str):
            text = outputs["output"]
        else:
            msgs = outputs.get("messages") or []
            if msgs:
                last = msgs[-1]
                content = last.get("content") if isinstance(last, dict) else None
                if isinstance(content, str):
                    text = content
                elif isinstance(content, list):
                    text = " ".join(
                        p.get("text", "") for p in content if isinstance(p, dict)
                    )
    text = text or ""

    # 1) Did the assistant return a substantive answer?
    answered = 1 if len(text.strip()) >= 20 else 0

    # 2) Did the "act" step fire anywhere in the trace's tool usage?
    #    We look for the tool name in the serialized run text.
    blob = str(run).lower()
    opened_case = 1 if "open_hr_case" in blob else 0

    return {"answered": answered, "opened_case": opened_case}
"""

online_rule = create_run_rule(
    client,
    project_name="hr-agent-workshop-william-reinhard",
    display_name="hr-online-code-check",
    sampling_rate=1.0,
    # Score only root traces, not every child LLM/tool/middleware span.
    filter="eq(is_root, true)",
    code_evaluator={"function_name": "perform_eval", "code": code_evaluator_source},
)

print("Rule ID:", online_rule["id"])
print("Open in UI:", online_rule["url"])

e3272f09-bf53-4cd5-a37c-93bd272d3078
Rule ID: 9e05479d-474f-493b-9ab7-08b8aa360479
Open in UI: https://smith.langchain.com/o/e3272f09-bf53-4cd5-a37c-93bd272d3078/projects/p/2c1c82b4-9604-4900-abbd-ccb46f5354b9/rules


## Annotation Queues — Close the Loop

Once runs have **feedback scores** (from the online code eval above, or any other source), route the ones that need a human to review.

LangSmith's annotation queues are that queue. We use **the same `create_run_rule` helper** — this time with `add_to_annotation_queue_id` set instead of an evaluator.
Any run matching the filter is added to the queue automatically. Here we route runs the code eval flagged as **not answered** (`answered == 0`).

In [20]:
queue = get_or_create_annotation_queue(
    client,
    name="hr-agent-needs-review",
    description="Runs routed here by the hr-agent's correctness automation rule.",
)
print(f"Queue: {queue.name} (id={queue.id})")


Queue: hr-agent-needs-review (id=37d54185-8261-4064-9117-ad8f1104c2e5)


In [28]:
# Same helper, no evaluator this time -- just a routing rule.
# Filter: root traces the code eval flagged with answered == 0 (no substantive answer).
queue_rule = create_run_rule(
    client,
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    display_name="hr-route-opened",
    sampling_rate=1.0,
    filter=(
        'and('
        'eq(is_root, true), '
        'eq(feedback_key, "opened_case"), '
        'eq(feedback_score, 1)'
        ')'
    ),
    add_to_annotation_queue_id=queue.id,
)

print("Queue rule ID:", queue_rule["id"])
print("Open in UI:    ", queue_rule["url"])

e3272f09-bf53-4cd5-a37c-93bd272d3078
Queue rule ID: 1dbb6c18-b234-4352-9cae-35ae0d8eb797
Open in UI:     https://smith.langchain.com/o/e3272f09-bf53-4cd5-a37c-93bd272d3078/projects/p/2c1c82b4-9604-4900-abbd-ccb46f5354b9/rules


### Trigger both rules

Both rules are live. Run a few more light traces and you'll see:

1. The online code eval fires on each new trace and attaches `answered` and `opened_case` feedback scores (~30s delay).
2. The queue rule fires on each *new feedback* that matches its filter (`answered == 0`) and routes the run to the review queue.

In [29]:
trigger_prompts = [
    # "Is employee 1001 paid inside band for their level?",
    # "What is the comp band for Software Engineering L3?",
    # "Who reports to manager 1201?",
    "Review manager 1062's team for anyone below band mid. For the most underpaid report, open a comp_review case"
]

time.sleep(100)

total = 0.0
for q in trigger_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].text[:120]}")

# Use the tenant_id LangSmith returned with the rule so the link works regardless of workspace.
tenant_id = queue_rule["payload"]["tenant_id"]

print(f"\nTotal: {total:.1f}s.")
print(f"\nOnline eval rule:  {online_rule['url']}")
print(f"Queue rule:        {queue_rule['url']}")
print(f"Queue (review UI): https://smith.langchain.com/o/{tenant_id}/annotation-queues/{queue.id}")
print("\nFeedback shows up in the rule pages within ~30s; queue placements follow once feedback lands.")

[ 4.2s] 1. **Status:** Manager ID 1062 was not found, so their team could not be reviewed.  
2. **Band assessment:** Not availab

Total: 4.2s.

Online eval rule:  https://smith.langchain.com/o/e3272f09-bf53-4cd5-a37c-93bd272d3078/projects/p/2c1c82b4-9604-4900-abbd-ccb46f5354b9/rules
Queue rule:        https://smith.langchain.com/o/e3272f09-bf53-4cd5-a37c-93bd272d3078/projects/p/2c1c82b4-9604-4900-abbd-ccb46f5354b9/rules
Queue (review UI): https://smith.langchain.com/o/e3272f09-bf53-4cd5-a37c-93bd272d3078/annotation-queues/37d54185-8261-4064-9117-ad8f1104c2e5

Feedback shows up in the rule pages within ~30s; queue placements follow once feedback lands.


### Common run-rule patterns

Swap the `filter` to build different rules:

| Use Case | `filter` |
|---|---|
| Unanswered runs (code eval) | `and(eq(feedback_key, "answered"), eq(feedback_score, 0))` |
| Errored runs | `eq(status, "error")` |
| Slow runs | `gt(latency, 10)` |
| Long-running tool calls | `and(eq(run_type, "tool"), gt(latency, 3))` |
| Case-opening runs (the "act" step) | `eq(name, "open_hr_case")` |
| 👎 from a user | `and(eq(feedback_key, "user_thumb"), eq(feedback_score, 0))` |

Both rule types — online eval and queue routing — go through the same `create_run_rule` helper. Use `delete_run_rule(client, rule_id)` to tear them down when you're done.

## Application Feedback — 👍 / 👎 + comment

Automated evaluators aren't the only source of feedback. Your **application** can attach a user's thumbs-up/down plus a free-text comment directly to the run — the minimal, production path for capturing real user signal alongside traces.

Two steps:
1. When you invoke the agent, capture the run id (via a `run_id` in config, or `collect_runs` / the LangSmith `RunTree`).
2. Call `client.create_feedback(run_id, key=..., score=..., comment=...)` when the user clicks 👍/👎.

The feedback lands on that exact run and shows up in three places in LangSmith: on the **trace's Feedback tab**, as a **`user_thumb` column** in the project's runs table, and in **filters** (`eq(feedback_key, "user_thumb")`) so you can route 👎 runs to a review queue with the same `create_run_rule` helper.

In [27]:
# 1. Invoke the agent and capture its run id. Passing an explicit run_id in the
#    config is the simplest path: your app already has it to attach feedback later.
feedback_run_id = str(uuid7())
cfg = {"configurable": {"thread_id": str(uuid7())}, "run_id": feedback_run_id}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Is employee 1005 inside band?"}]},
    config=cfg,
)
print("Agent:", result["messages"][-1].text[:160])
print("run_id:", feedback_run_id)

# 2. When the user clicks 👍/👎 in your UI, attach it to that run.
#    score: 1.0 for 👍, 0.0 for 👎.  key groups the feedback in the UI/filters.
user_clicked_thumbs_up = True
client.create_feedback(
    run_id=feedback_run_id,
    key="user_thumb",
    score=1.0 if user_clicked_thumbs_up else 0.0,
    comment="Clear answer with the band figures." if user_clicked_thumbs_up else "Missed the band.",
)
print("\nAttached 'user_thumb' feedback to the run.")
print("See it on the trace's Feedback tab, as a 'user_thumb' column in the project, "
      "and via filter eq(feedback_key, \"user_thumb\").")

Agent: **Status:** No employee profile found for ID 1005.  
**Salary vs. band:** Unable to assess without a valid profile and salary band.  
**Recommended next step:**
run_id: 01a04534-14d2-74f1-a468-5137db1622a4

Attached 'user_thumb' feedback to the run.
See it on the trace's Feedback tab, as a 'user_thumb' column in the project, and via filter eq(feedback_key, "user_thumb").


/var/folders/hn/rsmhv0g11zs_mjhdvxhxxr280000gn/T/ipykernel_29952/3736416686.py:15: LangSmithWarning: Creating feedback for a run without session_id is deprecated and will stop working in a future release. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#feedback-create
  client.create_feedback(


## Recap

| Part | What | API |
|---|---|---|
| **1. Prompt engineering** | Author, version, and share prompts; pull them into code | `client.push_prompt(...)` / `client.pull_prompt(...)` |
| **Warm-up** | Generate a few traces with the Module 1 agent | `agent.invoke(...)` |
| **2. Tracing + querying** | Auto-capture every run; pull back by filter | `LANGSMITH_TRACING=true`, `client.list_runs(filter=...)` |
| **3. Offline evals** | Score on demand against a dataset | `model.with_structured_output(...)` + `client.evaluate` |
| **4. Online evals** | Score every new trace automatically (code evaluator — no model secret needed) | `create_run_rule(..., code_evaluator_source=...)` |
| **Annotation queues** | Route flagged runs for human review | `create_run_rule(..., add_to_annotation_queue_id=...)` |

The full loop: trace → online eval scores it → run rule routes low scores to the queue → human reviews → fixes flow into the next dataset.

**Next:** Module 5 — **Engine** automates this entire loop (detect → diagnose → PR → evaluator) on your deployed agent.